# 03. Sparse Table and Sqrt Decomposition

Two more tools for **range queries**: one for **static** data (no updates) with O(1) queries, one simple technique that handles updates with O(√n).

**Learning order:** SPARSE TABLE → BUILD → QUERY → SQRT DECOMPOSITION → COMPARE → INTERVIEW THINKING

## 1. Sparse Table (static range min/max)

`table[k][i]` = the minimum of the segment starting at `i` of length `2^k`. Build from the previous level: `table[k][i] = min(table[k-1][i], table[k-1][i + 2^(k-1)])`.

**Query `[l, r]`:** let `k = floor(log2(r - l + 1))`. The two segments of length `2^k` starting at `l` and ending at `r` **overlap** and cover the range. For `min`/`max`/`gcd` overlap does not matter (idempotent), so the query is **O(1)**.

Build O(n log n), memory O(n log n). **No updates.**

In [1]:
class SparseTable:
    def __init__(self, data, op=min):
        self.op = op
        n = len(data)
        self.log = [0] * (n + 1)
        for i in range(2, n + 1):
            self.log[i] = self.log[i // 2] + 1
        self.table = [list(data)]
        k = 1
        while (1 << k) <= n:
            prev = self.table[-1]
            half = 1 << (k - 1)
            self.table.append([op(prev[i], prev[i + half]) for i in range(n - (1 << k) + 1)])
            k += 1

    def query(self, l, r):                                 # inclusive
        k = self.log[r - l + 1]
        return self.op(self.table[k][l], self.table[k][r - (1 << k) + 1])

st = SparseTable([2, 5, 1, 4, 9, 3])
print(st.query(0, 5), st.query(1, 3), st.query(3, 4))   # 1 1 4
mx = SparseTable([2, 5, 1, 4, 9, 3], max)
print(mx.query(0, 3), mx.query(2, 5))   # 5 9

1 1 4
5 9


## 2. Sqrt Decomposition

Split the array into blocks of size `B ≈ √n`. Keep the sum of each block.
- **Update:** change the element and its block's sum: O(1).
- **Range query:** partial blocks at both ends element by element (≤ 2B) + whole blocks in the middle (≤ n / B): **O(√n)**.

Simple, flexible (works for strange operations like "count of distinct", "mode") and the basis of **Mo's algorithm** for offline queries.

In [2]:
import math

class SqrtDecomposition:
    def __init__(self, data):
        self.data = data[:]
        self.block = max(1, int(math.sqrt(len(data))))
        self.sums = [0] * ((len(data) + self.block - 1) // self.block)
        for i, v in enumerate(data):
            self.sums[i // self.block] += v

    def update(self, i, value):
        self.sums[i // self.block] += value - self.data[i]
        self.data[i] = value

    def query(self, l, r):                                 # inclusive
        total, i = 0, l
        while i <= r and i % self.block != 0:              # left partial block
            total += self.data[i]; i += 1
        while i + self.block - 1 <= r:                     # whole blocks
            total += self.sums[i // self.block]; i += self.block
        while i <= r:                                      # right partial block
            total += self.data[i]; i += 1
        return total

sd = SqrtDecomposition([1, 2, 3, 4, 5, 6, 7, 8, 9])
print(sd.query(2, 7))   # 33
sd.update(4, 15)
print(sd.query(2, 7))   # 43

33
43


## 3. Comparison

| Structure | Build | Query | Update | Notes |
|---|---|---|---|---|
| Prefix sum | O(n) | O(1) | O(n) | sum only |
| Sparse table | O(n log n) | **O(1)** | none | min/max/gcd, static |
| Fenwick | O(n) | O(log n) | O(log n) | invertible ops |
| Segment tree | O(n) | O(log n) | O(log n) | any associative op, lazy range updates |
| Sqrt decomposition | O(n) | O(√n) | O(1) | simple, flexible |

## 4. Interview thinking

**Say:** "The data is static and I need many range minimum queries, so a sparse table gives O(1) per query after O(n log n) preprocessing." / "Updates are needed too, so a segment tree gives O(log n)."

**Common mistakes:** sparse table with a non-idempotent operation like sum (overlap double-counts), precomputing `log` incorrectly, off-by-one in the second block of the query.

## Quick Revision

- Sparse table: `table[k][i]` over `2^k`, query with two overlapping blocks, only for idempotent ops, no updates.
- Sqrt decomposition: blocks of √n, update O(1), query O(√n).
- Choose by: static vs dynamic, sum vs min/max, update type.